# NUS ST3236 — Stochastic Processes I
## Tutor-Style Kaggle Case Study with Interactive Bokeh Visualisations

### Case study: Delhi daily climate as an empirical Markov chain

We use Kaggle's **Daily Climate time series data**:

- Kaggle handle: `sumanthvrao/daily-climate-time-series-data`
- Period: 2013–2017
- Variables: `meantemp`, `humidity`, `wind_speed`, `meanpressure`
- Dataset page: https://www.kaggle.com/datasets/sumanthvrao/daily-climate-time-series-data

The aim is not to claim that weather is literally a finite homogeneous Markov chain. Instead, we will:

1. construct a finite-state stochastic process from continuous observations,
2. estimate a transition matrix,
3. study \(P^n\),
4. identify communicating classes,
5. examine irreducibility, recurrence and periodicity,
6. calculate a stationary distribution,
7. visualise convergence,
8. solve first-passage / hitting-time problems,
9. construct an absorbing-chain fundamental matrix,
10. simulate sample paths,
11. validate the first-order Markov approximation,
12. diagnose time-homogeneity using seasonal submodels.

This notebook is designed to connect **ST3236 theory ↔ numerical algorithms ↔ real data**.

# How to use this notebook as a new student

For every major step, the notebook follows the same learning cycle:

### A. Question
What stochastic-process question are we trying to answer?

### B. Mathematics
Which probability definition or theorem provides the answer?

### C. Code
How do we compute it from real data?

### D. Read the output
What do the numbers, rows, columns, curves, or graph structures mean?

### E. Inference
What conclusion is justified — and what conclusion would be too strong?

---

## A very important distinction

Throughout this notebook we will distinguish:

\[
\boxed{\text{observed data}}
\qquad\text{from}\qquad
\boxed{\text{fitted Markov model}}.
\]

For example, a stationary distribution is a mathematical property of the fitted transition matrix. It does **not automatically prove** that Delhi's physical climate process is stationary.

# Beginner vocabulary before we start

| Term | Plain-language meaning |
|---|---|
| **state** | the information we keep about the system at one time |
| **state space \(S\)** | all possible states |
| **stochastic process** | random variables indexed by time |
| **transition** | movement from today's state to tomorrow's state |
| **transition probability** | probability of a particular next state |
| **transition matrix \(P\)** | all one-step transition probabilities |
| **\(P^n\)** | probabilities \(n\) steps into the future |
| **communicate** | each state can eventually reach the other |
| **irreducible** | every state communicates with every other |
| **recurrent** | eventually returns with probability 1 |
| **transient** | may leave and never return |
| **period** | cycle structure of possible return times |
| **stationary distribution** | distribution unchanged by multiplication by \(P\) |
| **hitting time** | first time a target state is reached |
| **absorbing state** | once entered, it cannot be left |

# Learning map

The mathematical spine of the notebook is:

\[
\{X_t\}
\rightarrow
\hat P
\rightarrow
\hat P^n
\rightarrow
\text{state classification}
\rightarrow
\pi
\rightarrow
\text{long-run behaviour}
\]

with a second branch:

\[
\text{first-step analysis}
\rightarrow
\text{hitting times}
\rightarrow
\text{absorbing chains}.
\]

### Computational complexity

If there are \(T\) observations and \(K\) states:

| Operation | Typical complexity |
|---|---:|
| Count observed transitions | \(\Theta(T)\) |
| Row-normalise counts | \(\Theta(K^2)\) |
| Dense \(P^n\) using repeated squaring | \(O(K^3\log n)\) |
| Dense eigen decomposition for \(\pi\) | \(O(K^3)\) |
| Dense hitting-time linear solve | \(O(K^3)\) |
| SCC decomposition of state graph | \(\Theta(V+E)\) |

For the 3-state model here, these computations are tiny; the value is conceptual.

# 1. Environment setup

In [1]:
%pip install -q pandas numpy scipy bokeh networkx kagglehub

Note: you may need to restart the kernel to use updated packages.


### What happened in this step?

This cell installs the libraries used later.

There is **no stochastic-process inference yet**. Think of this as preparing the laboratory:

- `pandas` stores and transforms the time-series data.
- `numpy` performs matrix calculations such as \(P^n\).
- `networkx` interprets the Markov chain as a directed graph.
- `bokeh` produces interactive visualisations.
- `kagglehub` obtains the real dataset.

**Beginner habit:** distinguish *environment/setup cells* from *analysis cells*. Setup cells make later analysis possible but do not themselves support conclusions about the data.

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from functools import reduce
from math import gcd
from pathlib import Path
from typing import Iterable

import kagglehub
import networkx as nx
import numpy as np
import pandas as pd

from bokeh.io import output_notebook, show
from bokeh.layouts import column, gridplot
from bokeh.models import BasicTicker, ColorBar, ColumnDataSource, HoverTool, LinearColorMapper
from bokeh.palettes import Category10, Viridis256
from bokeh.plotting import figure
from bokeh.transform import factor_cmap

output_notebook(hide_banner=True)

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

### Why these imports matter

We import tools according to their role:

- **data layer:** `pandas`, `numpy`
- **graph layer:** `networkx`
- **visual layer:** Bokeh
- **software-design layer:** `dataclass`
- **data-access layer:** `kagglehub`

`output_notebook()` tells Bokeh to render interactively inside Jupyter.

**Inference:** none yet. At this point we have only defined our computational toolbox.

# 2. Download and load the Kaggle data

`kagglehub` can download the public dataset directly.

If your environment blocks Kaggle access, manually download:

- `DailyDelhiClimateTrain.csv`
- `DailyDelhiClimateTest.csv`

and place them under `./data/`.

We preserve chronological ordering because stochastic transitions depend on temporal adjacency.

In [3]:
DATASET_HANDLE = "sumanthvrao/daily-climate-time-series-data"

def locate_dataset_files() -> tuple[Path, Path]:
    try:
        root = Path(kagglehub.dataset_download(DATASET_HANDLE))
    except Exception as exc:
        root = Path("./data")
        print("Automatic Kaggle download failed; trying ./data")
        print("Reason:", repr(exc))

    train_candidates = list(root.rglob("DailyDelhiClimateTrain.csv"))
    test_candidates = list(root.rglob("DailyDelhiClimateTest.csv"))

    if not train_candidates or not test_candidates:
        raise FileNotFoundError(
            "Could not find the Kaggle CSV files. Download them from the dataset "
            "page and place them under ./data/."
        )

    return train_candidates[0], test_candidates[0]

TRAIN_PATH, TEST_PATH = locate_dataset_files()
print("Train:", TRAIN_PATH)
print("Test :", TEST_PATH)

Train: /home/anirban/.cache/kagglehub/datasets/sumanthvrao/daily-climate-time-series-data/versions/3/DailyDelhiClimateTrain.csv
Test : /home/anirban/.cache/kagglehub/datasets/sumanthvrao/daily-climate-time-series-data/versions/3/DailyDelhiClimateTest.csv


### How to read this output

The printed paths tell you where KaggleHub cached the two CSV files.

The important modelling fact is that the data already come with a **chronological train/test split**. We will fit transition probabilities on the earlier training period and later evaluate them on unseen future dates.

**Inference:** a temporal split is preferable to random shuffling because stochastic-process models must respect time order.

In [4]:
def load_climate_csv(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, parse_dates=["date"])
    return (
        df.sort_values("date")
          .drop_duplicates("date")
          .reset_index(drop=True)
    )

train_raw = load_climate_csv(TRAIN_PATH)
test_raw = load_climate_csv(TEST_PATH)

expected_cols = {"date", "meantemp", "humidity", "wind_speed", "meanpressure"}
assert expected_cols.issubset(train_raw.columns)
assert expected_cols.issubset(test_raw.columns)

display(train_raw.head())
display(test_raw.head())

print("Train shape:", train_raw.shape)
print("Test shape :", test_raw.shape)

,date,meantemp,humidity,wind_speed,meanpressure
0,2013-01-01,10.0000,84.5000,0.0000,"1,015.6667"
1,2013-01-02,7.4000,92.0000,2.9800,"1,017.8000"
2,2013-01-03,7.1667,87.0000,4.6333,"1,018.6667"
3,2013-01-04,8.6667,71.3333,1.2333,"1,017.1667"
4,2013-01-05,6.0000,86.8333,3.7000,"1,016.5000"


,date,meantemp,humidity,wind_speed,meanpressure
0,2017-01-01,15.9130,85.8696,2.7435,59.0000
1,2017-01-02,18.5000,77.2222,2.8944,"1,018.2778"
2,2017-01-03,17.1111,81.8889,4.0167,"1,018.3333"
3,2017-01-04,18.7000,70.0500,4.5450,"1,015.7000"
4,2017-01-05,18.3889,74.9444,3.3000,"1,014.3333"


Train shape: (1462, 5)
Test shape : (114, 5)


### What did we learn from the loaded tables?

Check four things in the displayed output:

1. `date` should be parsed as a date rather than plain text.
2. Rows should be ordered chronologically.
3. The expected climate variables should be present.
4. Train and test should contain plausible numbers of observations.

At this stage we are checking **data integrity**, not Markov properties.

**Inference:** if chronology or columns were wrong, every subsequent transition probability would be unreliable.

## Tutor checkpoint: why no random shuffle?

In ordinary tabular ML we often shuffle rows.

For a stochastic process, however,

\[
X_t\rightarrow X_{t+1}
\]

is defined by chronology.

Shuffling destroys the transition structure and turns real one-day transitions into meaningless pairs.

In [5]:
full_raw = (
    pd.concat(
        [
            train_raw.assign(split="train"),
            test_raw.assign(split="test"),
        ],
        ignore_index=True,
    )
    .sort_values("date")
    .reset_index(drop=True)
)

display(full_raw.describe().T)

full_raw["day_gap"] = full_raw["date"].diff().dt.days

print("Date range:", full_raw["date"].min().date(), "to", full_raw["date"].max().date())
print("Duplicate dates:", full_raw["date"].duplicated().sum())
print("\nMissing values:")
display(full_raw.isna().sum().to_frame("missing"))

print("\nObserved calendar gaps:")
display(full_raw["day_gap"].value_counts(dropna=False).sort_index().to_frame("count"))

,count,mean,min,25%,50%,75%,max,std
date,1576,2015-02-27 10:15:50.253807104,2013-01-01 00:00:00,2014-01-29 18:00:00,2015-02-27 12:00:00,2016-03-27 06:00:00,2017-04-24 00:00:00,NaN
meantemp,"1,576.0000",25.2219,6.0000,18.5000,27.1667,31.1429,38.7143,7.3450
humidity,"1,576.0000",60.4452,13.4286,49.7500,62.4405,72.1250,100.0000,16.9800
wind_speed,"1,576.0000",6.8993,0.0000,3.7000,6.3636,9.2625,42.2200,4.5107
meanpressure,"1,576.0000","1,010.5932",-3.0417,"1,001.8750","1,009.0556","1,015.2000","7,679.3333",175.2427


Date range: 2013-01-01 to 2017-04-24
Duplicate dates: 1

Missing values:


,missing
date,0
meantemp,0
humidity,0
wind_speed,0
meanpressure,0
split,0
day_gap,1



Observed calendar gaps:


,count
day_gap,
0.0000,1
1.0000,1574
NaN,1


### How to interpret the quality checks

Focus on:

- **duplicate dates:** ideally zero;
- **missing values:** determine whether preprocessing is required;
- **day gaps:** a value of 1 means two consecutive rows are truly consecutive days.

A Markov step in this notebook is defined as **one day**. If a row gap is 2 or 3 days, treating it as one step would mix different time horizons.

**Inference:** only one-calendar-day pairs should contribute to the one-step transition matrix.

## Why inspect date gaps?

A one-step transition should represent one day.

If two consecutive rows are separated by several days and we count them as one transition, then

\[
X_t\to X_{t+1}
\]

no longer means "today to tomorrow".

Our transition estimator will therefore retain only adjacent rows separated by exactly one calendar day.

# 3. Interactive Bokeh exploration

In [6]:
source = ColumnDataSource(full_raw)

p_temp = figure(
    width=950,
    height=300,
    x_axis_type="datetime",
    title="Delhi daily mean temperature",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
p_temp.line("date", "meantemp", source=source, line_width=1.5)
p_temp.add_tools(HoverTool(
    tooltips=[("date", "@date{%F}"), ("mean temperature", "@meantemp{0.0} °C")],
    formatters={"@date": "datetime"},
))

p_hum = figure(
    width=950,
    height=300,
    x_axis_type="datetime",
    x_range=p_temp.x_range,
    title="Delhi daily humidity",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)
p_hum.line("date", "humidity", source=source, line_width=1.5)
p_hum.add_tools(HoverTool(
    tooltips=[("date", "@date{%F}"), ("humidity", "@humidity{0.0}")],
    formatters={"@date": "datetime"},
))

show(column(p_temp, p_hum))

### How to inspect these plots

Use Bokeh's hover and zoom tools.

Look for:

- repeating annual temperature cycles;
- gradual seasonal movement rather than random jumps;
- humidity patterns that may carry information not contained in temperature alone.

### Inference

The visible seasonality warns us that the assumption

\[
P(X_{t+1}=j\mid X_t=i)
\]

being identical throughout the year may be imperfect.

That does **not** make Markov chains useless. It gives us a hypothesis to test later: *is one global transition matrix too simple?*

### What should you notice?

Temperature exhibits strong seasonality.

That immediately gives us a modelling warning:

\[
P(X_{t+1}=j\mid X_t=i)
\]

may depend on the time of year.

Later we will compare quarter-specific transition matrices against the global transition matrix.

# 4. Define a finite state space

The raw temperature variable is continuous.

ST3236's finite-state Markov machinery becomes directly applicable after defining:

\[
S=\{\text{Cold},\text{Mild},\text{Hot}\}.
\]

We determine the cut-points from the **Kaggle training period only** to avoid test leakage.

The boundaries are the 1/3 and 2/3 quantiles:

\[
q_{1/3},\qquad q_{2/3}.
\]

The names `Cold`, `Mild`, and `Hot` are relative to this dataset; they are not universal meteorological definitions.

In [7]:
STATE_ORDER = ["Cold", "Mild", "Hot"]

q_low, q_high = train_raw["meantemp"].quantile([1/3, 2/3]).to_numpy()
print(f"Training cut points: {q_low:.3f} °C and {q_high:.3f} °C")

def assign_temperature_state(
    values: pd.Series,
    low: float = q_low,
    high: float = q_high,
) -> pd.Categorical:
    return pd.cut(
        values,
        bins=[-np.inf, low, high, np.inf],
        labels=STATE_ORDER,
        ordered=True,
        include_lowest=True,
    )

train = train_raw.copy()
test = test_raw.copy()
full = full_raw.copy()

for df in (train, test, full):
    df["state"] = assign_temperature_state(df["meantemp"]).astype(str)

display(train[["date", "meantemp", "state"]].head(12))
display(train["state"].value_counts().reindex(STATE_ORDER).to_frame("train_count"))

Training cut points: 22.429 °C and 30.500 °C


,date,meantemp,state
0,2013-01-01,10.0000,Cold
1,2013-01-02,7.4000,Cold
2,2013-01-03,7.1667,Cold
3,2013-01-04,8.6667,Cold
4,2013-01-05,6.0000,Cold
5,2013-01-06,7.0000,Cold
6,2013-01-07,7.0000,Cold
7,2013-01-08,8.8571,Cold
8,2013-01-09,14.0000,Cold
9,2013-01-10,11.0000,Cold


,train_count
state,
Cold,487
Mild,500
Hot,475


### How to read the state construction

The two printed cut points divide training temperatures into three roughly populated regimes.

For example, if the boundaries were approximately \(20^\circ C\) and \(29^\circ C\):

- below the first threshold → `Cold`,
- between thresholds → `Mild`,
- above the second threshold → `Hot`.

### Why fit thresholds on training data only?

Using test data to choose the thresholds would leak future information into model construction.

### Inference

We have converted a continuous process into a finite-state process:

\[
X_t\in\{\text{Cold},\text{Mild},\text{Hot}\}.
\]

This **state definition is a modelling choice**, not a fact supplied by nature.

In [8]:
state_source = ColumnDataSource(full)

p_state = figure(
    width=950,
    height=320,
    x_axis_type="datetime",
    y_range=STATE_ORDER,
    title="Continuous temperature mapped into discrete Markov states",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p_state.scatter(
    x="date",
    y="state",
    source=state_source,
    size=5,
    alpha=0.65,
    color=factor_cmap("state", palette=Category10[3], factors=STATE_ORDER),
)

p_state.add_tools(HoverTool(
    tooltips=[
        ("date", "@date{%F}"),
        ("temperature", "@meantemp{0.0} °C"),
        ("state", "@state"),
    ],
    formatters={"@date": "datetime"},
))

show(p_state)

### How to read the state plot

Each dot is one day's discretised state.

Notice that states tend to persist for runs of consecutive days. Long runs imply relatively large diagonal transition probabilities such as

\[
p_{\text{Hot,Hot}}.
\]

### Inference

The visible persistence suggests that tomorrow's state is related to today's state. That is exactly the kind of temporal dependence a Markov model is designed to represent.

# 5. Estimate the one-step transition matrix

For a homogeneous discrete-time Markov chain,

\[
p_{ij}
=
P(X_{t+1}=j\mid X_t=i).
\]

Given transition counts \(N_{ij}\), the maximum-likelihood estimator is

\[
\hat p_{ij}
=
\frac{N_{ij}}{\sum_k N_{ik}}.
\]

So the familiar row-normalised count table is not an arbitrary trick: it is the MLE of the transition probabilities.

In [9]:
def transition_counts(
    df: pd.DataFrame,
    states: list[str],
    state_col: str = "state",
) -> pd.DataFrame:
    current = df.iloc[:-1].reset_index(drop=True)
    nxt = df.iloc[1:].reset_index(drop=True)

    valid = (nxt["date"] - current["date"]).dt.days.eq(1)

    pairs = pd.DataFrame({
        "current": current[state_col],
        "next": nxt[state_col],
        "valid": valid,
    }).loc[lambda x: x["valid"]]

    counts = pd.crosstab(pairs["current"], pairs["next"])

    return counts.reindex(index=states, columns=states, fill_value=0)

def counts_to_transition_matrix(counts: pd.DataFrame) -> pd.DataFrame:
    denominators = counts.sum(axis=1).replace(0, np.nan)
    return counts.div(denominators, axis=0).fillna(0.0)

train_counts = transition_counts(train, STATE_ORDER)
train_P_df = counts_to_transition_matrix(train_counts)

print("Observed one-day transition counts")
display(train_counts)

print("Estimated transition probabilities")
display(train_P_df)

assert np.allclose(train_P_df.sum(axis=1), 1.0)

Observed one-day transition counts


next,Cold,Mild,Hot
current,,,
Cold,464,22,0
Mild,22,398,80
Hot,0,80,395


Estimated transition probabilities


next,Cold,Mild,Hot
current,,,
Cold,0.9547,0.0453,0.0000
Mild,0.0440,0.7960,0.1600
Hot,0.0000,0.1684,0.8316


### Reading the transition tables

The **count matrix** answers:

> How many times did we actually observe state \(i\) today and state \(j\) tomorrow?

The **probability matrix** divides each row by its total:

\[
\hat p_{ij}=\frac{N_{ij}}{\sum_k N_{ik}}.
\]

Each row must sum to 1 because it is a conditional probability distribution over tomorrow's possible states.

### Inference checklist

- A large diagonal entry → strong persistence.
- A small `Cold → Hot` probability → abrupt one-day jumps are uncommon.
- Different rows → tomorrow's distribution depends on today's state.

If all rows looked nearly identical, today's state would carry little predictive information.

## How to read \(P\)

Rows represent the current state; columns represent tomorrow's state.

For example,

\[
\hat p_{\text{Mild,Hot}}
\]

estimates

\[
P(X_{t+1}=\text{Hot}\mid X_t=\text{Mild}).
\]

A common exam/programming mistake is to reverse the direction.

In [10]:
def transition_heatmap(
    matrix: pd.DataFrame,
    title: str,
    width: int = 440,
    height: int = 380,
):
    long = (
        matrix.rename_axis("from_state")
              .reset_index()
              .melt("from_state", var_name="to_state", value_name="probability")
    )
    long["label"] = long["probability"].map(lambda x: f"{x:.3f}")
    src = ColumnDataSource(long)

    mapper = LinearColorMapper(
        palette=Viridis256,
        low=0.0,
        high=max(float(long["probability"].max()), 1e-12),
    )

    p = figure(
        x_range=list(matrix.columns),
        y_range=list(reversed(matrix.index.tolist())),
        width=width,
        height=height,
        title=title,
        tools="hover,save,reset",
    )

    p.rect(
        x="to_state",
        y="from_state",
        width=1,
        height=1,
        source=src,
        fill_color={"field": "probability", "transform": mapper},
        line_color=None,
    )

    p.text(
        x="to_state",
        y="from_state",
        text="label",
        source=src,
        text_align="center",
        text_baseline="middle",
    )

    p.add_tools(HoverTool(
        tooltips=[
            ("from", "@from_state"),
            ("to", "@to_state"),
            ("probability", "@probability{0.000}"),
        ]
    ))

    p.add_layout(ColorBar(color_mapper=mapper, ticker=BasicTicker()), "right")
    p.xaxis.axis_label = "Tomorrow"
    p.yaxis.axis_label = "Today"

    return p

show(transition_heatmap(train_P_df, "Estimated one-day transition matrix"))

### How to read the heatmap

Read **from row to column**:

\[
\text{today's state}\rightarrow\text{tomorrow's state}.
\]

The number inside a cell is the estimated one-step probability.

### Example interpretation

If the `Mild → Mild` cell is 0.80, then among training days classified as `Mild`, about 80% of valid next-day observations were also `Mild`.

### Inference

The heatmap is the empirical summary of the Markov transition mechanism. Strong values along the diagonal imply short-term state persistence.

# 6. A reusable `DiscreteMarkovChain` class

The class below packages the core algorithms:

- \(P^n\)
- stationary distribution
- communicating classes
- closed classes
- recurrence/transience classification for finite chains
- period estimation
- expected hitting times
- absorbing-state transformation
- simulation

This gives us a clean separation between probability logic and plotting code.

In [11]:
@dataclass(frozen=True)
class DiscreteMarkovChain:
    states: tuple[str, ...]
    P: np.ndarray

    def __post_init__(self):
        P = np.asarray(self.P, dtype=float)

        if P.shape != (len(self.states), len(self.states)):
            raise ValueError("P must be K x K, where K=len(states).")

        if np.any(P < -1e-12):
            raise ValueError("Transition probabilities cannot be negative.")

        if not np.allclose(P.sum(axis=1), 1.0):
            raise ValueError("Each transition-matrix row must sum to 1.")

        object.__setattr__(self, "P", P)

    @property
    def K(self) -> int:
        return len(self.states)

    def n_step(self, n: int) -> np.ndarray:
        if n < 0:
            raise ValueError("n must be non-negative.")
        return np.linalg.matrix_power(self.P, n)

    def stationary_distribution(self) -> np.ndarray:
        eigenvalues, eigenvectors = np.linalg.eig(self.P.T)
        idx = int(np.argmin(np.abs(eigenvalues - 1.0)))

        vec = np.real(eigenvectors[:, idx])

        if vec.sum() < 0:
            vec = -vec

        pi = vec / vec.sum()
        pi = np.clip(pi, 0.0, None)
        return pi / pi.sum()

    def graph(self, eps: float = 1e-12) -> nx.DiGraph:
        G = nx.DiGraph()
        G.add_nodes_from(range(self.K))

        for i in range(self.K):
            for j in range(self.K):
                if self.P[i, j] > eps:
                    G.add_edge(i, j, weight=float(self.P[i, j]))

        return G

    def classify_states(self, eps: float = 1e-12) -> dict:
        G = self.graph(eps)
        sccs = list(nx.strongly_connected_components(G))

        closed_classes = []
        transient_states = []

        for component in sccs:
            has_outgoing = any(
                u in component and v not in component
                for u, v in G.edges()
            )

            if has_outgoing:
                transient_states.extend(component)
            else:
                closed_classes.append(component)

        if closed_classes:
            recurrent_states = sorted(set().union(*closed_classes))
        else:
            recurrent_states = []

        return {
            "communicating_classes": sccs,
            "closed_classes": closed_classes,
            "recurrent_states": recurrent_states,
            "transient_states": sorted(transient_states),
            "irreducible": len(sccs) == 1,
        }

    def periods(self, max_n: int = 100, tol: float = 1e-12) -> dict[str, int]:
        # Inspect return times n with P^n(i,i)>0 and take their gcd.
        return_times = {i: [] for i in range(self.K)}
        power = np.eye(self.K)

        for n in range(1, max_n + 1):
            power = power @ self.P

            for i in range(self.K):
                if power[i, i] > tol:
                    return_times[i].append(n)

        result = {}

        for i, times in return_times.items():
            result[self.states[i]] = reduce(gcd, times) if times else 0

        return result

    def expected_hitting_times(self, target_state: str) -> pd.Series:
        target = self.states.index(target_state)
        non_target = [i for i in range(self.K) if i != target]

        Q = self.P[np.ix_(non_target, non_target)]
        A = np.eye(len(non_target)) - Q
        b = np.ones(len(non_target))

        h_non_target = np.linalg.solve(A, b)

        h = np.zeros(self.K)
        h[non_target] = h_non_target

        return pd.Series(
            h,
            index=self.states,
            name=f"E[T_{target_state}]",
        )

    def absorbing_version(self, target_state: str) -> "DiscreteMarkovChain":
        target = self.states.index(target_state)

        P_abs = self.P.copy()
        P_abs[target, :] = 0.0
        P_abs[target, target] = 1.0

        return DiscreteMarkovChain(self.states, P_abs)

    def simulate(
        self,
        n_steps: int,
        start_state: str,
        random_state: int = 42,
    ) -> list[str]:
        gen = np.random.default_rng(random_state)
        current = self.states.index(start_state)

        path = [current]

        for _ in range(n_steps):
            current = int(gen.choice(self.K, p=self.P[current]))
            path.append(current)

        return [self.states[i] for i in path]


mc = DiscreteMarkovChain(
    states=tuple(STATE_ORDER),
    P=train_P_df.to_numpy(),
)

mc

DiscreteMarkovChain(states=('Cold', 'Mild', 'Hot'), P=array([[0.95473251, 0.04526749, 0.        ],
       [0.044     , 0.796     , 0.16      ],
       [0.        , 0.16842105, 0.83157895]]))

### Why wrap the Markov chain in a class?

The class stores two things:

\[
(S,P)
\]

where:
- \(S\) is the ordered state set,
- \(P\) is the transition matrix.

Methods then answer different stochastic-process questions.

This is good software design because the mathematical object and operations on it stay together.

### Important validation

The constructor verifies that:

\[
p_{ij}\ge 0,\qquad \sum_jp_{ij}=1.
\]

If those conditions fail, the object is not a valid transition matrix.

### Inference

Creating `mc` does not add new evidence; it packages the estimated model so that later graph, hitting-time, stationary, and simulation analyses are consistent.

# NetworkX mini-tutorial — understanding a Markov chain as a graph

This section is deliberately detailed because graph language makes many ST3236 definitions much easier.

A finite Markov chain with matrix \(P\) can be represented as a **weighted directed graph**:

\[
G=(V,E).
\]

- one node \(v_i\in V\) for each Markov state \(i\);
- a directed edge \(i\to j\) whenever \(p_{ij}>0\);
- edge weight \(w_{ij}=p_{ij}\).

So probability theory and graph theory describe the same transition structure from different viewpoints.

| Markov-chain concept | NetworkX / graph concept |
|---|---|
| state | node |
| possible transition | directed edge |
| transition probability | edge weight |
| accessible state | reachable node |
| communicating states | mutually reachable nodes |
| communicating class | strongly connected component |
| irreducible chain | strongly connected graph |
| closed class | SCC with no outgoing edge |
| path | possible multi-step sequence of states |

In [12]:
G = mc.graph()

print("Python type:", type(G))
print("Directed?", G.is_directed())
print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())

state_name = {i: mc.states[i] for i in G.nodes}

print("\nNodes:")
for node in G.nodes:
    print(f"  node {node}: {state_name[node]}")

print("\nEdges with transition-probability weights:")
for u, v, attrs in G.edges(data=True):
    print(
        f"  {state_name[u]} -> {state_name[v]} "
        f"with p={attrs['weight']:.4f}"
    )

Python type: <class 'networkx.classes.digraph.DiGraph'>
Directed? True
Number of nodes: 3
Number of edges: 7

Nodes:
  node 0: Cold
  node 1: Mild
  node 2: Hot

Edges with transition-probability weights:
  Cold -> Cold with p=0.9547
  Cold -> Mild with p=0.0453
  Mild -> Cold with p=0.0440
  Mild -> Mild with p=0.7960
  Mild -> Hot with p=0.1600
  Hot -> Mild with p=0.1684
  Hot -> Hot with p=0.8316


## Reading the `DiGraph`

`nx.DiGraph` means **directed graph**.

Direction matters because

\[
p_{ij}
\]

and

\[
p_{ji}
\]

need not be equal.

The graph's edge attribute `weight` stores the transition probability.

### Inference

If an edge `Cold → Hot` does not exist, then the fitted empirical chain assigns zero probability to that direct one-step transition.

That does **not** imply `Hot` is unreachable from `Cold`; a multi-step path such as

\[
\text{Cold}\to\text{Mild}\to\text{Hot}
\]

may still exist.

## NetworkX capability 1 — adjacency and neighbourhood queries

Useful methods:

- `G.successors(i)` — states directly reachable in one step from \(i\);
- `G.predecessors(i)` — states that can directly transition into \(i\);
- `G.out_degree(i)` — number of outgoing graph edges;
- `G.in_degree(i)` — number of incoming graph edges;
- `G[i][j]["weight"]` — transition probability stored on edge \(i\to j\).

These are graph-structure queries; they are different from multiplying probabilities.

In [13]:
for i in G.nodes:
    succ = [state_name[j] for j in G.successors(i)]
    pred = [state_name[j] for j in G.predecessors(i)]

    print(f"{state_name[i]}")
    print("  successors   :", succ)
    print("  predecessors :", pred)
    print("  out-degree   :", G.out_degree(i))
    print("  in-degree    :", G.in_degree(i))

Cold
  successors   : ['Cold', 'Mild']
  predecessors : ['Cold', 'Mild']
  out-degree   : 2
  in-degree    : 2
Mild
  successors   : ['Cold', 'Mild', 'Hot']
  predecessors : ['Cold', 'Mild', 'Hot']
  out-degree   : 3
  in-degree    : 3
Hot
  successors   : ['Mild', 'Hot']
  predecessors : ['Mild', 'Hot']
  out-degree   : 2
  in-degree    : 2


### Inference from neighbourhoods

If each state lists all three states as successors, every destination has positive one-step probability from every origin.

But irreducibility does **not require** complete one-step connectivity.

A chain can still be irreducible when some one-step edges are missing, provided multi-step directed paths connect all pairs.

## NetworkX capability 2 — edge attributes

NetworkX lets an edge carry arbitrary metadata.

For this Markov graph we use:

```python
weight = transition_probability
```

In larger applications, edges could also store:
- counts,
- costs,
- labels,
- timestamps,
- capacities.

For a Markov chain, the most important attribute is probability.

In [14]:
edge_table = pd.DataFrame(
    [
        {
            "from": state_name[u],
            "to": state_name[v],
            "probability": attrs["weight"],
        }
        for u, v, attrs in G.edges(data=True)
    ]
).sort_values(["from", "probability"], ascending=[True, False])

display(edge_table)

,from,to,probability
0,Cold,Cold,0.9547
1,Cold,Mild,0.0453
6,Hot,Hot,0.8316
5,Hot,Mild,0.1684
3,Mild,Mild,0.7960
4,Mild,Hot,0.1600
2,Mild,Cold,0.0440


### How to read the edge table

Within each `from` state, probabilities should sum to 1 because they reproduce one row of \(P\).

This is a useful debugging technique: the graph and transition matrix should encode identical information.

In [15]:
edge_row_sums = (
    edge_table.groupby("from")["probability"]
              .sum()
              .reindex(STATE_ORDER)
)

display(edge_row_sums.to_frame("sum_of_outgoing_probabilities"))
assert np.allclose(edge_row_sums.to_numpy(), 1.0)

,sum_of_outgoing_probabilities
from,
Cold,1.0000
Mild,1.0000
Hot,1.0000


### Inference

The successful assertion confirms that converting \(P\) into a NetworkX graph preserved the stochastic row-sum constraint.

This is a **consistency check**, not a new scientific conclusion.

## NetworkX capability 3 — paths and reachability

A directed path

\[
i\to k_1\to\cdots\to j
\]

means there exists some possible sequence of transitions from \(i\) to \(j\).

Important functions:

- `nx.has_path(G, i, j)`
- `nx.shortest_path(G, i, j)`
- `nx.descendants(G, i)`
- `nx.ancestors(G, i)`

### Important probability distinction

`shortest_path` minimises the **number of edges** by default.

It does **not** return the most probable path.

In [16]:
for origin_name in STATE_ORDER:
    for destination_name in STATE_ORDER:
        i = STATE_ORDER.index(origin_name)
        j = STATE_ORDER.index(destination_name)

        reachable = nx.has_path(G, i, j)
        path = nx.shortest_path(G, i, j) if reachable else None
        named_path = [state_name[x] for x in path] if path else None

        print(
            f"{origin_name:>4} -> {destination_name:<4} "
            f"reachable={reachable!s:<5} shortest_path={named_path}"
        )

Cold -> Cold reachable=True  shortest_path=['Cold']
Cold -> Mild reachable=True  shortest_path=['Cold', 'Mild']
Cold -> Hot  reachable=True  shortest_path=['Cold', 'Mild', 'Hot']
Mild -> Cold reachable=True  shortest_path=['Mild', 'Cold']
Mild -> Mild reachable=True  shortest_path=['Mild']
Mild -> Hot  reachable=True  shortest_path=['Mild', 'Hot']
 Hot -> Cold reachable=True  shortest_path=['Hot', 'Mild', 'Cold']
 Hot -> Mild reachable=True  shortest_path=['Hot', 'Mild']
 Hot -> Hot  reachable=True  shortest_path=['Hot']


### Inference

If every ordered pair is reachable, then every state is **accessible** from every other state.

That is exactly the reachability condition behind irreducibility.

Notice how NetworkX lets us test an ST3236 definition using a standard graph algorithm.

## NetworkX capability 4 — strongly connected components

A **strongly connected component (SCC)** is a maximal set of nodes where every node can reach every other node.

This corresponds exactly to a **communicating class**.

Useful functions:

```python
nx.strongly_connected_components(G)
nx.is_strongly_connected(G)
nx.number_strongly_connected_components(G)
```

NetworkX uses efficient graph algorithms; SCC decomposition is linear in graph size:

\[
\Theta(V+E).
\]

In [17]:
sccs = list(nx.strongly_connected_components(G))

print("Number of SCCs:", len(sccs))
print(
    "SCCs:",
    [[state_name[i] for i in sorted(component)] for component in sccs],
)
print("Strongly connected?", nx.is_strongly_connected(G))

Number of SCCs: 1
SCCs: [['Cold', 'Mild', 'Hot']]
Strongly connected? True


### Stochastic-process inference

- one SCC containing every state → chain is **irreducible**;
- multiple SCCs → chain is **reducible**.

For our finite fitted chain, irreducibility has major consequences:

\[
\text{finite + irreducible}
\Rightarrow
\text{all states positive recurrent}
\]

and the stationary distribution is unique.

## NetworkX capability 5 — weak connectivity

`nx.is_weakly_connected(G)` ignores arrow direction when deciding whether the graph is connected.

This can be useful descriptively, but **weak connectivity is not enough for Markov irreducibility**.

Irreducibility requires **strong** connectivity because transitions have direction.

In [18]:
print("Weakly connected? :", nx.is_weakly_connected(G))
print("Strongly connected?:", nx.is_strongly_connected(G))

Weakly connected? : True
Strongly connected?: True


### Inference

A graph can be weakly connected but not strongly connected.

Therefore, when analysing communicating classes, always use the directed notion: **strong connectivity**.

## NetworkX capability 6 — cycles

A recurrent Markov state must be involved in return behaviour.

NetworkX can enumerate simple directed cycles using:

```python
nx.simple_cycles(G)
```

A self-loop such as

\[
\text{Mild}\to\text{Mild}
\]

is a cycle of length 1.

Cycle lengths are closely related to the Markov notion of **period**.

In [19]:
cycles = list(nx.simple_cycles(G))

named_cycles = [
    [state_name[i] for i in cycle]
    for cycle in cycles
]

print("Number of simple directed cycles:", len(named_cycles))
for cycle in named_cycles[:20]:
    print(cycle)

Number of simple directed cycles: 5
['Cold']
['Mild']
['Hot']
['Cold', 'Mild']
['Mild', 'Hot']


### Inference

If a state has a self-loop, return in one step is possible, so its period is 1.

For an irreducible chain, all states share the same period. Therefore one self-loop anywhere in the communicating class is enough to establish aperiodicity of the entire irreducible class.

## NetworkX capability 7 — condensation graph

When a chain is reducible, it can be compressed by replacing each communicating class with one node.

NetworkX calls this the **condensation graph**:

```python
nx.condensation(G)
```

The condensation graph is always a directed acyclic graph (DAG).

This is especially useful for identifying:
- closed communicating classes,
- transient regions,
- one-way flow between classes.

In [20]:
C = nx.condensation(G)

print("Condensation nodes:", C.number_of_nodes())
print("Condensation edges:", C.number_of_edges())
print("Is condensation a DAG?", nx.is_directed_acyclic_graph(C))

for node, attrs in C.nodes(data=True):
    members = [state_name[i] for i in sorted(attrs["members"])]
    print(f"Condensation node {node}: members={members}")

Condensation nodes: 1
Condensation edges: 0
Is condensation a DAG? True
Condensation node 0: members=['Cold', 'Mild', 'Hot']


### Inference

If the original graph is irreducible, it has only one communicating class, so the condensation graph collapses to a single node.

For a reducible chain, sink nodes of the condensation graph correspond to **closed communicating classes**, which are recurrent classes in a finite chain.

## NetworkX capability 8 — matrix conversion

NetworkX can reconstruct a weighted adjacency matrix:

```python
nx.to_numpy_array(...)
```

For our graph, that matrix should equal \(P\) when nodes are supplied in the same order.

In [21]:
P_from_graph = nx.to_numpy_array(
    G,
    nodelist=list(range(mc.K)),
    weight="weight",
)

print("P from original Markov model:")
print(mc.P)

print("\nWeighted adjacency matrix reconstructed by NetworkX:")
print(P_from_graph)

print("\nmax absolute difference =", np.max(np.abs(mc.P - P_from_graph)))

assert np.allclose(mc.P, P_from_graph)

P from original Markov model:
[[0.95473251 0.04526749 0.        ]
 [0.044      0.796      0.16      ]
 [0.         0.16842105 0.83157895]]

Weighted adjacency matrix reconstructed by NetworkX:
[[0.95473251 0.04526749 0.        ]
 [0.044      0.796      0.16      ]
 [0.         0.16842105 0.83157895]]

max absolute difference = 0.0


### Inference

The exact agreement demonstrates an important equivalence:

\[
\boxed{\text{transition matrix}}
\quad\Longleftrightarrow\quad
\boxed{\text{weighted directed graph}}.
\]

Matrix methods are usually convenient for probabilities; graph methods are usually convenient for reachability and structural classification.

# Visualising the Markov graph with NetworkX layout + Bokeh

NetworkX supplies graph layout algorithms such as:

- `circular_layout`
- `spring_layout`
- `kamada_kawai_layout`
- `spectral_layout`

A layout assigns each node an \((x,y)\) coordinate for drawing.

**Important:** layout coordinates have no stochastic meaning. They only improve readability.

In [22]:
layout = nx.circular_layout(G)

node_rows = []
for i, (x, y) in layout.items():
    node_rows.append({
        "node": i,
        "state": state_name[i],
        "x": float(x),
        "y": float(y),
    })

edge_rows = []
for u, v, attrs in G.edges(data=True):
    x0, y0 = layout[u]
    x1, y1 = layout[v]

    # Slightly trim the line visually so labels are easier to read.
    edge_rows.append({
        "from": state_name[u],
        "to": state_name[v],
        "x0": float(x0),
        "y0": float(y0),
        "x1": float(x1),
        "y1": float(y1),
        "probability": float(attrs["weight"]),
        "label_x": float((x0 + x1) / 2),
        "label_y": float((y0 + y1) / 2),
        "label": f"{attrs['weight']:.2f}",
    })

node_plot_df = pd.DataFrame(node_rows)
edge_plot_df = pd.DataFrame(edge_rows)

display(node_plot_df)
display(edge_plot_df)

,node,state,x,y
0,0,Cold,1.0000,0.0000
1,1,Mild,-0.5000,0.8660
2,2,Hot,-0.5000,-0.8660


,from,to,x0,y0,x1,y1,probability,label_x,label_y,label
0,Cold,Cold,1.0000,0.0000,1.0000,0.0000,0.9547,1.0000,0.0000,0.95
1,Cold,Mild,1.0000,0.0000,-0.5000,0.8660,0.0453,0.2500,0.4330,0.05
2,Mild,Cold,-0.5000,0.8660,1.0000,0.0000,0.0440,0.2500,0.4330,0.04
3,Mild,Mild,-0.5000,0.8660,-0.5000,0.8660,0.7960,-0.5000,0.8660,0.80
4,Mild,Hot,-0.5000,0.8660,-0.5000,-0.8660,0.1600,-0.5000,-0.0000,0.16
5,Hot,Mild,-0.5000,-0.8660,-0.5000,0.8660,0.1684,-0.5000,-0.0000,0.17
6,Hot,Hot,-0.5000,-0.8660,-0.5000,-0.8660,0.8316,-0.5000,-0.8660,0.83


### What is being prepared?

`nx.circular_layout(G)` computes display coordinates.

We then create two plotting tables:

1. node positions,
2. edge start/end positions and transition probabilities.

The edge labels make the visual graph directly comparable to the transition matrix.

In [23]:
from bokeh.models import Arrow, NormalHead, Label

p_graph = figure(
    width=750,
    height=600,
    title="Markov transition graph — NetworkX structure, Bokeh rendering",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

# Draw directed arrows.
for row_data in edge_plot_df.to_dict("records"):
    p_graph.add_layout(
        Arrow(
            end=NormalHead(size=9),
            x_start=row_data["x0"],
            y_start=row_data["y0"],
            x_end=row_data["x1"],
            y_end=row_data["y1"],
            line_alpha=0.55,
        )
    )

    p_graph.add_layout(
        Label(
            x=row_data["label_x"],
            y=row_data["label_y"],
            text=row_data["label"],
            text_font_size="10pt",
        )
    )

node_source = ColumnDataSource(node_plot_df)

p_graph.scatter(
    x="x",
    y="y",
    size=55,
    source=node_source,
)

for row_data in node_plot_df.to_dict("records"):
    p_graph.add_layout(
        Label(
            x=row_data["x"],
            y=row_data["y"],
            text=row_data["state"],
            text_align="center",
            text_baseline="middle",
            text_font_size="11pt",
        )
    )

p_graph.add_tools(
    HoverTool(
        tooltips=[("state", "@state")],
        renderers=[p_graph.renderers[-1]],
    )
)

p_graph.axis.visible = False
p_graph.grid.visible = False

show(p_graph)

## How to read the graph visual

- **node** = climate state;
- **arrow** = possible one-day transition;
- **number on arrow** = estimated transition probability;
- **self-loop probability** appears in the transition matrix even if a compact drawing makes it visually less prominent.

### Why use both matrix and graph views?

The matrix is better for:
- exact probability values,
- matrix multiplication,
- \(P^n\),
- stationary calculations.

The graph is better for:
- reachability,
- communicating classes,
- closed classes,
- structural understanding.

A strong ST3236 student should be comfortable moving between both representations.

# NetworkX capabilities beyond this case study

NetworkX is a general graph-analysis library. The same `Graph` / `DiGraph` objects support:

### Connectivity and traversal
- BFS / DFS
- ancestors / descendants
- connected components
- strongly connected components
- topological sorting for DAGs

### Paths
- shortest paths
- weighted shortest paths
- all simple paths
- path existence

### Structural measures
- degree
- density
- clustering
- assortativity

### Centrality
- PageRank
- betweenness centrality
- closeness centrality
- eigenvector centrality

### Community/network algorithms
- communities
- cuts
- flows
- matching
- spanning trees

### Matrix interoperability
- adjacency matrices
- Laplacians
- NumPy/SciPy conversion

For ST3236, the most relevant subset is:

\[
\boxed{
\text{directed edges}
+
\text{reachability}
+
\text{SCCs}
+
\text{cycles}
+
\text{weighted adjacency}
}.
\]

Do not apply generic graph measures automatically just because NetworkX offers them. Use a measure only when it corresponds to a meaningful stochastic-process question.

# 7. Chapman–Kolmogorov and multi-step probabilities

For a time-homogeneous Markov chain,

\[
P^{(m+n)}=P^mP^n.
\]

Thus

\[
(P^n)_{ij}
=
P(X_{t+n}=j\mid X_t=i).
\]

We compare 1-, 2-, 7-, and 30-day transition matrices.

In [24]:
horizons = [1, 2, 7, 30]
figs = []

for n in horizons:
    Pn_df = pd.DataFrame(
        mc.n_step(n),
        index=STATE_ORDER,
        columns=STATE_ORDER,
    )

    figs.append(
        transition_heatmap(
            Pn_df,
            title=f"{n}-day transition matrix",
            width=410,
            height=350,
        )
    )

show(gridplot([[figs[0], figs[1]], [figs[2], figs[3]]]))

### How to compare the four heatmaps

- \(P\): tomorrow.
- \(P^2\): two days ahead.
- \(P^7\): one week ahead.
- \(P^{30}\): roughly one month ahead.

As \(n\) grows, compare the **rows**.

### Inference

If rows become increasingly similar, knowledge of the starting state matters less at longer horizons.

For an irreducible aperiodic finite chain,

\[
P^n(i,\cdot)\to\pi
\]

for every starting state \(i\).

So similar long-horizon rows are visual evidence of model convergence toward a common limiting distribution.

### What should happen as \(n\) grows?

If the fitted chain is irreducible and aperiodic, then

\[
P^n(i,j)\to\pi_j.
\]

That means every row of \(P^n\) approaches the same vector.

Intuitively, the chain gradually forgets where it started.

# 8. Communicating classes and irreducibility

Interpret the transition matrix as a directed graph:

\[
i\to j
\quad\Longleftrightarrow\quad
p_{ij}>0.
\]

Two states communicate if each is reachable from the other.

Graph-theoretically, the communicating classes are **strongly connected components**.

For a finite chain:
- states in closed communicating classes are recurrent,
- states outside closed classes are transient.

In [25]:
classification = mc.classify_states()

def names(indices: Iterable[int]) -> list[str]:
    return [mc.states[i] for i in indices]

print(
    "Communicating classes:",
    [names(sorted(c)) for c in classification["communicating_classes"]],
)
print(
    "Closed classes:",
    [names(sorted(c)) for c in classification["closed_classes"]],
)
print("Recurrent states:", names(classification["recurrent_states"]))
print("Transient states:", names(classification["transient_states"]))
print("Irreducible:", classification["irreducible"])

Communicating classes: [['Cold', 'Mild', 'Hot']]
Closed classes: [['Cold', 'Mild', 'Hot']]
Recurrent states: ['Cold', 'Mild', 'Hot']
Transient states: []
Irreducible: True


### How to interpret the classification output

A **communicating class** is a collection of states that can all reach one another.

If all three states appear in one class, the empirical state graph is strongly connected.

If `Irreducible: True`, then:

\[
\forall i,j,\quad i\leftrightarrow j.
\]

### Inference for a finite chain

When a finite chain is irreducible:

- every state is recurrent,
- every state is positive recurrent,
- a unique stationary distribution exists.

This is a theorem about the fitted chain, not necessarily a claim that real weather is globally stationary.

### The finite-chain theorem to remember

A **finite irreducible** Markov chain has:
- all states positive recurrent,
- a unique stationary distribution.

Aperiodicity is additionally important when we want ordinary convergence of \(P^n\).

# 9. Periodicity

For a state \(i\),

\[
d(i)
=
\gcd\{n\ge1:P^n(i,i)>0\}.
\]

If \(d(i)=1\), state \(i\) is aperiodic.

In an irreducible chain, all states have the same period.

In [26]:
periods = mc.periods(max_n=60)
periods

{'Cold': 1, 'Mild': 1, 'Hot': 1}

### How to read the period result

A period of 1 means **aperiodic**.

Why might period 1 appear here? If a state can remain unchanged for one step, then

\[
p_{ii}>0,
\]

so returning to \(i\) in exactly one step is possible. The gcd of possible return times therefore includes 1.

### Inference

If the fitted chain is both irreducible and aperiodic, the usual finite-state convergence theorem applies:

\[
P^n(i,j)\to\pi_j.
\]

A positive self-transition probability \(p_{ii}>0\) gives an immediate one-step return, so that state's period is 1.

This is why empirical Markov chains with strong state persistence are often aperiodic.

# 10. Stationary distribution

A stationary distribution \(\pi\) satisfies

\[
\pi P=\pi,
\qquad
\sum_i\pi_i=1.
\]

Equivalently,

\[
P^\top \pi^\top=\pi^\top,
\]

so \(\pi^\top\) is an eigenvector of \(P^\top\) associated with eigenvalue 1.

In [27]:
pi = mc.stationary_distribution()

stationary_df = pd.DataFrame({
    "state": STATE_ORDER,
    "stationary_probability": pi,
})

display(stationary_df)

print("πP:")
display(pd.DataFrame([pi @ mc.P], columns=STATE_ORDER, index=["πP"]))

print("max |πP - π| =", np.max(np.abs(pi @ mc.P - pi)))

,state,stationary_probability
0,Cold,0.3326
1,Mild,0.3422
2,Hot,0.3251


πP:


,Cold,Mild,Hot
πP,0.3326,0.3422,0.3251


max |πP - π| = 5.551115123125783e-17


### How to read \(\pi\)

The displayed vector contains one long-run model probability per state.

The numerical check

\[
\pi P\approx \pi
\]

should have an error extremely close to floating-point zero.

### Inference

Within the fitted homogeneous Markov model, \(\pi_i\) is the long-run fraction of time allocated to state \(i\).

Be careful: **stationary distribution of the model** does not prove the original climate series is stationary.

In [28]:
p_pi = figure(
    x_range=STATE_ORDER,
    width=650,
    height=360,
    title="Estimated stationary distribution",
    tools="hover,save,reset",
)

p_pi.vbar(
    x="state",
    top="stationary_probability",
    width=0.65,
    source=ColumnDataSource(stationary_df),
)

p_pi.y_range.start = 0
p_pi.yaxis.axis_label = "Stationary probability"
p_pi.add_tools(HoverTool(
    tooltips=[("state", "@state"), ("π", "@stationary_probability{0.000}")]
))

show(p_pi)

### Reading the stationary-distribution plot

The bar heights are the entries of \(\pi\).

A taller bar means that, under repeated application of the fitted transition mechanism, the chain spends a larger long-run proportion of steps in that state.

### Inference

This plot summarises long-run model occupancy, whereas the original time-series plot showed the actual historically observed chronology. Those are related but conceptually different objects.

## Interpretation and caution

Within the fitted homogeneous Markov model, \(\pi_i\) is the long-run proportion of time spent in state \(i\).

But this is a statement about the **model**.

The observed climate process has seasonality; therefore the real data-generating process does not have to be stationary just because the fitted matrix possesses a stationary distribution.

# 11. Convergence to stationarity

Total variation distance between two discrete distributions is

\[
d_{\mathrm{TV}}(p,q)
=
\frac12\sum_j |p_j-q_j|.
\]

For each possible starting state \(i\), we compute

\[
d_{\mathrm{TV}}(e_iP^n,\pi)
\]

over time.

This directly measures how rapidly the chain forgets its initial state.

In [29]:
def convergence_frame(
    chain: DiscreteMarkovChain,
    max_n: int = 90,
) -> pd.DataFrame:
    pi = chain.stationary_distribution()
    rows = []

    for start_idx, start_state in enumerate(chain.states):
        initial = np.zeros(chain.K)
        initial[start_idx] = 1.0

        distribution = initial.copy()

        for n in range(max_n + 1):
            tv = 0.5 * np.abs(distribution - pi).sum()

            rows.append({
                "start_state": start_state,
                "n": n,
                "tv_distance": tv,
            })

            distribution = distribution @ chain.P

    return pd.DataFrame(rows)

conv = convergence_frame(mc, max_n=90)

p_conv = figure(
    width=850,
    height=420,
    title="Convergence toward the stationary distribution",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

for state in STATE_ORDER:
    part = conv.loc[conv["start_state"] == state]

    p_conv.line(
        part["n"],
        part["tv_distance"],
        line_width=2,
        legend_label=state,
    )

p_conv.xaxis.axis_label = "n days"
p_conv.yaxis.axis_label = "Total variation distance to π"
p_conv.legend.title = "Initial state"

show(p_conv)

### How to read total-variation convergence

For each possible starting state, the curve begins some distance away from stationarity and should move toward zero.

\[
d_{\mathrm{TV}}(e_iP^n,\pi)=0
\]

would mean the \(n\)-step distribution is exactly \(\pi\).

### Inference

The rate at which the curves fall tells us how rapidly the fitted chain **forgets its initial state**.

This is closely related to the idea of **mixing time** used in Markov-chain Monte Carlo.

# 12. First-passage / hitting times

Let

\[
T_j=\inf\{n\ge0:X_n=j\}.
\]

Define

\[
h_i=E_i[T_j].
\]

First-step analysis yields, for \(i\ne j\),

\[
h_i
=
1+\sum_{k\ne j}p_{ik}h_k,
\]

with

\[
h_j=0.
\]

In matrix form:

\[
(I-Q)h=\mathbf 1.
\]

We will use `Hot` as the target state.

In [30]:
hot_hitting = mc.expected_hitting_times("Hot")
display(hot_hitting.to_frame())

,E[T_Hot]
Cold,34.4159
Mild,12.3250
Hot,0.0000


### How to read expected hitting times

For each starting state \(i\), the output estimates

\[
E_i[T_{\text{Hot}}].
\]

The target itself has value zero because if we start in `Hot`, the hitting time defined with \(n\ge0\) is already 0.

### Inference

A larger expected value means the fitted chain typically needs more transitions to reach `Hot` from that starting state.

This is not a temperature forecast in physical units; it is a property of the discretised Markov model.

In [31]:
hit_df = hot_hitting.rename("expected_days").reset_index()
hit_df.columns = ["start_state", "expected_days"]

p_hit = figure(
    x_range=STATE_ORDER,
    width=650,
    height=360,
    title="Expected first-passage time to Hot",
    tools="hover,save,reset",
)

p_hit.vbar(
    x="start_state",
    top="expected_days",
    width=0.65,
    source=ColumnDataSource(hit_df),
)

p_hit.y_range.start = 0
p_hit.yaxis.axis_label = "Expected days"

p_hit.add_tools(HoverTool(
    tooltips=[
        ("start", "@start_state"),
        ("E[T_Hot]", "@expected_days{0.00} days"),
    ]
))

show(p_hit)

### What the hitting-time chart adds

The table gives exact computed values; the chart makes relative magnitude easier to compare.

### Inference

If `Cold` has a much larger bar than `Mild`, the transition structure makes reaching `Hot` substantially slower when starting from `Cold`.

# 13. Distribution of a hitting time

The mean alone does not describe uncertainty.

We also want

\[
P_i(T_{\text{Hot}}\le n).
\]

A useful trick is to modify the transition matrix so that `Hot` becomes absorbing.

Then the probability mass accumulated in `Hot` after \(n\) steps is exactly the probability of having hit `Hot` by time \(n\).

In [32]:
mc_hot_abs = mc.absorbing_version("Hot")
hot_idx = STATE_ORDER.index("Hot")

rows = []

for start_state in ["Cold", "Mild"]:
    start_idx = STATE_ORDER.index(start_state)

    distribution = np.zeros(mc.K)
    distribution[start_idx] = 1.0

    for n in range(61):
        rows.append({
            "start_state": start_state,
            "n": n,
            "cdf": distribution[hot_idx],
        })

        distribution = distribution @ mc_hot_abs.P

hitting_cdf = pd.DataFrame(rows)

p_cdf = figure(
    width=850,
    height=420,
    title="CDF of first-passage time to Hot",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

for state in ["Cold", "Mild"]:
    part = hitting_cdf.loc[hitting_cdf["start_state"] == state]

    p_cdf.line(
        part["n"],
        part["cdf"],
        line_width=2,
        legend_label=f"start={state}",
    )

p_cdf.xaxis.axis_label = "n days"
p_cdf.yaxis.axis_label = "P(T_Hot ≤ n)"
p_cdf.legend.location = "bottom_right"

show(p_cdf)

### Why make `Hot` absorbing?

We modify the chain so that once it reaches `Hot` it remains there:

\[
p_{\text{Hot,Hot}}=1.
\]

Then the probability mass in `Hot` after \(n\) steps equals

\[
P(T_{\text{Hot}}\le n).
\]

### How to read the CDF

At day \(n\), the curve height is the probability that `Hot` has been reached **at least once** by then.

### Inference

This gives much richer information than a single expected hitting time because it describes the whole distribution of waiting time.

# 14. Absorbing chains and the fundamental matrix

For an absorbing chain, after separating transient from absorbing states,

\[
P=
\begin{bmatrix}
Q&R\\
0&I
\end{bmatrix}.
\]

The fundamental matrix is

\[
N=(I-Q)^{-1}.
\]

Its entry \(N_{ij}\) is the expected number of visits to transient state \(j\) before absorption, starting from transient state \(i\).

The vector

\[
N\mathbf 1
\]

contains expected times to absorption.

In [33]:
target_idx = STATE_ORDER.index("Hot")
transient_idx = [i for i in range(mc.K) if i != target_idx]
transient_states = [STATE_ORDER[i] for i in transient_idx]

Q = mc_hot_abs.P[np.ix_(transient_idx, transient_idx)]
N = np.linalg.inv(np.eye(len(transient_idx)) - Q)

N_df = pd.DataFrame(
    N,
    index=[f"start={s}" for s in transient_states],
    columns=[f"visits_to={s}" for s in transient_states],
)

display(N_df)

expected_absorption = N @ np.ones(len(transient_idx))

absorption_df = pd.Series(
    expected_absorption,
    index=transient_states,
    name="expected_days_to_Hot",
).to_frame()

display(absorption_df)

for state, expected in zip(transient_states, expected_absorption):
    assert np.isclose(expected, hot_hitting[state])

print("Fundamental-matrix and first-step calculations agree.")

,visits_to=Cold,visits_to=Mild
start=Cold,28.1659,6.2500
start=Mild,6.0750,6.2500


,expected_days_to_Hot
Cold,34.4159
Mild,12.3250


Fundamental-matrix and first-step calculations agree.


### Reading the fundamental matrix

For transient states,

\[
N=(I-Q)^{-1}.
\]

Entry \(N_{ij}\) means:

> expected number of visits to transient state \(j\) before absorption, given that we started in transient state \(i\).

The row sum

\[
N\mathbf 1
\]

is expected time until absorption.

### Inference

The equality check confirms that the **fundamental-matrix method** and **first-step linear equations** are two mathematically equivalent ways to solve the same hitting-time problem.

# 15. Mean recurrence time

For an irreducible positive recurrent chain,

\[
E_i[T_i^+]=\frac{1}{\pi_i}.
\]

This gives a beautiful connection between:
- long-run occupancy,
- return-time behaviour.

We compare \(1/\pi_i\) with empirical gaps between repeated observations of each state.

In [34]:
def empirical_mean_return_gap(
    df: pd.DataFrame,
    state: str,
) -> float:
    positions = np.flatnonzero(df["state"].to_numpy() == state)

    if len(positions) < 2:
        return np.nan

    return float(np.diff(positions).mean())

return_comparison = pd.DataFrame({
    "state": STATE_ORDER,
    "model_1_over_pi": 1.0 / pi,
    "empirical_mean_gap_full_data": [
        empirical_mean_return_gap(full, s)
        for s in STATE_ORDER
    ],
})

return_comparison["difference"] = (
    return_comparison["empirical_mean_gap_full_data"]
    - return_comparison["model_1_over_pi"]
)

display(return_comparison)

,state,model_1_over_pi,empirical_mean_gap_full_data,difference
0,Cold,3.0062,2.7644,-0.2418
1,Mild,2.9220,2.8409,-0.0811
2,Hot,3.0758,3.0061,-0.0697


### Comparing theory with empirical return gaps

For the fitted irreducible positive recurrent chain,

\[
E_i[T_i^+]=\frac1{\pi_i}.
\]

The empirical column measures average observed spacing between occurrences of the same state in the historical data.

### Inference

Close agreement would support the stationary-chain approximation.

Differences are informative rather than automatically "errors": they can reflect seasonality, finite-sample noise, or failure of the homogeneous first-order assumptions.

## Why can empirical and theoretical return times differ?

The theorem applies exactly to the fitted stationary Markov model.

Real-data discrepancies can arise from:
- finite sample size,
- seasonality,
- non-homogeneous transitions,
- insufficient state information,
- imperfect first-order Markov behaviour.

This distinction between **theorem under assumptions** and **empirical model adequacy** is crucial.

# 16. Diagnose the first-order Markov assumption

The first-order Markov property is

\[
P(X_{t+1}\mid X_t,X_{t-1},\ldots)
=
P(X_{t+1}\mid X_t).
\]

We compare held-out predictive log loss for:

1. **zero-order model**
   \[
   P(X_{t+1})
   \]

2. **first-order model**
   \[
   P(X_{t+1}\mid X_t)
   \]

3. **second-order model**
   \[
   P(X_{t+1}\mid X_t,X_{t-1})
   \]

If second-order conditioning materially improves held-out likelihood, then the current state alone is not capturing all useful temporal information.

In [35]:
STATE_TO_IDX = {s: i for i, s in enumerate(STATE_ORDER)}

def state_indices(series: pd.Series) -> np.ndarray:
    return series.map(STATE_TO_IDX).to_numpy(dtype=int)

train_idx = state_indices(train["state"])
test_idx = state_indices(test["state"])

zero_probs = (
    train["state"]
    .value_counts(normalize=True)
    .reindex(STATE_ORDER)
    .to_numpy()
)

first_probs = mc.P

def fit_second_order(
    indices: np.ndarray,
    K: int,
    laplace: float = 1.0,
) -> np.ndarray:
    counts = np.full((K, K, K), laplace, dtype=float)

    for a, b, c in zip(indices[:-2], indices[1:-1], indices[2:]):
        counts[a, b, c] += 1.0

    return counts / counts.sum(axis=2, keepdims=True)

second_probs = fit_second_order(train_idx, mc.K)

def mean_nll_zero(indices: np.ndarray, probs: np.ndarray) -> float:
    p = probs[indices]
    return float(-np.mean(np.log(np.clip(p, 1e-15, 1.0))))

def mean_nll_first(indices: np.ndarray, P: np.ndarray) -> float:
    current = indices[:-1]
    nxt = indices[1:]
    p = P[current, nxt]
    return float(-np.mean(np.log(np.clip(p, 1e-15, 1.0))))

def mean_nll_second(indices: np.ndarray, tensor: np.ndarray) -> float:
    prev = indices[:-2]
    current = indices[1:-1]
    nxt = indices[2:]
    p = tensor[prev, current, nxt]
    return float(-np.mean(np.log(np.clip(p, 1e-15, 1.0))))

diagnostic_metrics = pd.DataFrame({
    "model": [
        "Zero-order: P(X[t+1])",
        "First-order: P(X[t+1] | X[t])",
        "Second-order: P(X[t+1] | X[t], X[t-1])",
    ],
    "test_mean_negative_log_likelihood": [
        mean_nll_zero(test_idx[1:], zero_probs),
        mean_nll_first(test_idx, first_probs),
        mean_nll_second(test_idx, second_probs),
    ],
})

display(diagnostic_metrics)

,model,test_mean_negative_log_likelihood
0,Zero-order: P(X[t+1]),1.0958
1,First-order: P(X[t+1] | X[t]),0.4750
2,"Second-order: P(X[t+1] | X[t], X[t-1])",0.4099


### Why compare zero-, first-, and second-order models?

We are asking how much history is useful.

- zero-order ignores today's state;
- first-order uses \(X_t\);
- second-order uses \((X_{t-1},X_t)\).

The score is held-out **negative log likelihood (NLL)**:

\[
-\frac1N\sum_t\log \hat P(X_{t+1}\mid\text{history}).
\]

Lower is better.

### Inference

- first-order < zero-order NLL → today's state carries useful information;
- second-order < first-order NLL → one lag may not contain all predictive information.

That suggests either a higher-order chain or a better state representation.

In [36]:
p_diag = figure(
    y_range=list(reversed(diagnostic_metrics["model"].tolist())),
    width=900,
    height=340,
    title="Held-out diagnostic of Markov memory length",
    tools="hover,save,reset",
)

p_diag.hbar(
    y="model",
    right="test_mean_negative_log_likelihood",
    height=0.55,
    source=ColumnDataSource(diagnostic_metrics),
)

p_diag.xaxis.axis_label = "Mean negative log likelihood — lower is better"

p_diag.add_tools(HoverTool(
    tooltips=[
        ("model", "@model"),
        ("NLL", "@test_mean_negative_log_likelihood{0.0000}"),
    ]
))

show(p_diag)

### Reading the NLL chart

Shorter bars are better because lower negative log likelihood means the model assigned higher probability to the outcomes that actually occurred.

### Inference

Do not select a more complicated Markov order simply because it fits training data better. We are comparing on the chronological **test period**, which helps detect whether extra memory actually generalises.

## How to interpret this diagnostic

### First-order beats zero-order
The current state contains useful predictive information.

### Second-order beats first-order
The preceding state contains additional information not fully encoded in \(X_t\).

This does not automatically mean "always use a second-order chain".

A better alternative may be to improve the state:

\[
Z_t=(X_t,\text{season},\text{humidity regime},\ldots).
\]

The Markov property is fundamentally a statement about whether the **state representation is sufficient**.

# 17. Diagnose time homogeneity

A homogeneous Markov chain assumes the transition rule does not explicitly depend on \(t\):

\[
P(X_{t+1}=j\mid X_t=i)=p_{ij}.
\]

Because weather is seasonal, we compare transition matrices across calendar quarters.

In [37]:
def transition_matrix_for_subset(
    df: pd.DataFrame,
    current_row_mask: pd.Series,
) -> pd.DataFrame:
    current = df.iloc[:-1].reset_index(drop=True)
    nxt = df.iloc[1:].reset_index(drop=True)

    valid = (
        (nxt["date"] - current["date"]).dt.days.eq(1)
        & current_row_mask.iloc[:-1].reset_index(drop=True)
    )

    pairs = pd.DataFrame({
        "current": current["state"],
        "next": nxt["state"],
    }).loc[valid]

    counts = pd.crosstab(pairs["current"], pairs["next"])
    counts = counts.reindex(index=STATE_ORDER, columns=STATE_ORDER, fill_value=0)

    return counts_to_transition_matrix(counts)

train = train.copy()
train["quarter"] = train["date"].dt.quarter

quarter_matrices = {}

for q in sorted(train["quarter"].unique()):
    quarter_matrices[q] = transition_matrix_for_subset(
        train,
        train["quarter"].eq(q),
    )

quarter_figs = [
    transition_heatmap(
        quarter_matrices[q],
        title=f"Quarter {q}",
        width=390,
        height=340,
    )
    for q in sorted(quarter_matrices)
]

show(gridplot([
    [quarter_figs[0], quarter_figs[1]],
    [quarter_figs[2], quarter_figs[3]],
]))

### Why estimate quarter-specific matrices?

Time homogeneity assumes a single \(P\) works at every date.

Here we estimate:

\[
P_{Q1},P_{Q2},P_{Q3},P_{Q4}.
\]

### How to infer from the heatmaps

If the same row changes substantially across quarters, then tomorrow's conditional state distribution depends on the season even after conditioning on today's state.

That is evidence against a single perfectly homogeneous transition mechanism.

In [38]:
global_P = train_P_df.to_numpy()

homogeneity_rows = []

for q, Pq_df in quarter_matrices.items():
    distance = np.linalg.norm(Pq_df.to_numpy() - global_P, ord="fro")

    homogeneity_rows.append({
        "quarter": f"Q{q}",
        "frobenius_distance_from_global_P": distance,
    })

homogeneity_df = pd.DataFrame(homogeneity_rows)
display(homogeneity_df)

p_hom = figure(
    x_range=homogeneity_df["quarter"].tolist(),
    width=650,
    height=350,
    title="Quarter-specific transition matrices vs global P",
    tools="hover,save,reset",
)

p_hom.vbar(
    x="quarter",
    top="frobenius_distance_from_global_P",
    width=0.6,
    source=ColumnDataSource(homogeneity_df),
)

p_hom.y_range.start = 0
p_hom.yaxis.axis_label = "Frobenius distance"

show(p_hom)

,quarter,frobenius_distance_from_global_P
0,Q1,0.8677
1,Q2,1.3598
2,Q3,0.9791
3,Q4,0.2665


### What is Frobenius distance doing?

For matrices \(A\) and \(B\),

\[
\|A-B\|_F
=
\sqrt{\sum_{i,j}(a_{ij}-b_{ij})^2}.
\]

A larger distance means a quarter-specific transition matrix is more different from the global matrix.

### Inference

This is a descriptive diagnostic, not a formal hypothesis test.

Large distances strengthen the case for:
- seasonal transition matrices, or
- augmenting the state with season.

## Interpretation

Large quarter-to-quarter differences indicate that a single global \(P\) is averaging across different regimes.

Two natural extensions are:

### Non-homogeneous Markov chain
Use a time-dependent transition matrix \(P_t\).

### State augmentation
Define

\[
Z_t=(X_t,\text{quarter}_t).
\]

A richer state can make the process closer to a homogeneous first-order Markov chain.

# 18. Out-of-sample next-state prediction

The Kaggle split is chronological, which is ideal for validation.

Given current state \(i\), the first-order Markov classifier predicts

\[
\hat X_{t+1}
=
\arg\max_j p_{ij}.
\]

We compare this with an IID baseline that always predicts the most common training state.

In [39]:
current = test_idx[:-1]
actual = test_idx[1:]

markov_pred = mc.P[current].argmax(axis=1)

markov_accuracy = float(np.mean(markov_pred == actual))

iid_pred = int(np.argmax(zero_probs))
iid_accuracy = float(np.mean(actual == iid_pred))

evaluation = pd.DataFrame({
    "model": ["IID baseline", "First-order Markov"],
    "next_state_accuracy": [iid_accuracy, markov_accuracy],
})

display(evaluation)

,model,next_state_accuracy
0,IID baseline,0.2566
1,First-order Markov,0.8407


### Interpreting predictive accuracy

The IID baseline ignores today's state.

The Markov model predicts tomorrow using the most probable destination in the row corresponding to today's state.

### Inference

If Markov accuracy exceeds the IID baseline, the current state has practical next-step predictive value.

However, accuracy discards probability calibration, so NLL remains the more natural score for a stochastic transition model.

### Accuracy is secondary to probabilistic scoring

A transition matrix predicts a probability distribution, not merely a class.

Negative log likelihood uses the full predictive distribution:

\[
-\frac1N\sum_t
\log P(X_{t+1}^{\text{observed}}\mid X_t).
\]

This is usually more informative than accuracy when evaluating stochastic transition models.

# 19. Simulate a Markov sample path

Once \(P\) is estimated, we can generate

\[
X_0,X_1,\ldots,X_n
\]

by sampling each next state from the row of \(P\) corresponding to the current state.

Simulation connects theoretical Markov chains to Monte Carlo analysis.

In [40]:
simulated_states = mc.simulate(
    n_steps=365,
    start_state="Mild",
    random_state=RANDOM_STATE,
)

sim_df = pd.DataFrame({
    "step": np.arange(len(simulated_states)),
    "state": simulated_states,
})

p_sim = figure(
    width=900,
    height=330,
    y_range=STATE_ORDER,
    title="One simulated 365-step state path",
    tools="pan,wheel_zoom,box_zoom,reset,save",
)

p_sim.scatter(
    x="step",
    y="state",
    source=ColumnDataSource(sim_df),
    size=6,
    alpha=0.7,
    color=factor_cmap("state", palette=Category10[3], factors=STATE_ORDER),
)

p_sim.xaxis.axis_label = "Simulation step"

show(p_sim)

### What does the simulated path mean?

This is **not another observed weather series**.

It is one random trajectory generated entirely from the estimated matrix \(P\).

Each step samples tomorrow's state from the appropriate transition-probability row.

### Inference

Simulation shows the behavioural implications of the fitted model: persistence, transitions, run lengths, and long-run occupancy.

# 20. Monte Carlo verification of \(\pi\)

For an irreducible, aperiodic finite chain, a sufficiently long sample path should spend approximately a fraction \(\pi_i\) of its time in state \(i\).

Let's simulate 100,000 transitions and compare empirical frequencies with the theoretical stationary distribution.

In [41]:
long_path = mc.simulate(
    n_steps=100_000,
    start_state="Cold",
    random_state=RANDOM_STATE,
)

simulation_pi = (
    pd.Series(long_path)
    .value_counts(normalize=True)
    .reindex(STATE_ORDER)
    .to_numpy()
)

mc_check = pd.DataFrame({
    "state": STATE_ORDER,
    "theoretical_pi": pi,
    "simulation_frequency": simulation_pi,
})

mc_check["absolute_error"] = (
    mc_check["simulation_frequency"] - mc_check["theoretical_pi"]
).abs()

display(mc_check)

,state,theoretical_pi,simulation_frequency,absolute_error
0,Cold,0.3326,0.3318,0.0008
1,Mild,0.3422,0.3468,0.0045
2,Hot,0.3251,0.3214,0.0037


### Why simulate 100,000 steps?

The ergodic theorem suggests that, for an appropriate finite chain, long-run empirical state frequencies should approach \(\pi\).

### Inference

Small absolute errors between simulation frequencies and \(\pi\) are a numerical sanity check that:

- the simulator uses \(P\) correctly;
- the stationary calculation is consistent with long-run model behaviour.

In [42]:
mc_long = mc_check.melt(
    id_vars="state",
    value_vars=["theoretical_pi", "simulation_frequency"],
    var_name="quantity",
    value_name="probability",
)

mc_long["x"] = list(zip(mc_long["state"], mc_long["quantity"]))

factors = [
    (state, quantity)
    for state in STATE_ORDER
    for quantity in ["theoretical_pi", "simulation_frequency"]
]

p_mc = figure(
    x_range=factors,
    width=900,
    height=380,
    title="Stationary distribution vs long-run simulation",
    tools="hover,save,reset",
)

p_mc.vbar(
    x="x",
    top="probability",
    width=0.85,
    source=ColumnDataSource(mc_long),
)

p_mc.y_range.start = 0
p_mc.yaxis.axis_label = "Probability / frequency"
p_mc.xaxis.major_label_orientation = 0.8

show(p_mc)

ValueError: failed to validate FigureOptions(...).x_range: expected an element of either Instance(Range), Either(Tuple(Float, Float), Tuple(Datetime, Datetime), Tuple(TimeDelta, TimeDelta)), Seq(String), Object(Series) or Object(GroupBy), got [('Cold', 'theoretical_pi'), ('Cold', 'simulation_frequency'), ('Mild', 'theoretical_pi'), ('Mild', 'simulation_frequency'), ('Hot', 'theoretical_pi'), ('Hot', 'simulation_frequency')]

### Reading the comparison plot

For each state, compare:
- the theoretical stationary probability;
- the frequency observed in a very long simulated path.

The closer each pair is, the stronger the Monte Carlo confirmation of the stationary calculation.

# 21. Richer states: temperature + humidity

A temperature-only state may throw away predictive information.

We can define a richer process:

\[
Z_t
=
(\text{temperature state},\text{humidity regime}).
\]

This can make the state more informative, but increases model complexity.

With \(K\) states, a dense row-stochastic transition matrix has approximately

\[
K(K-1)
\]

free transition parameters.

So richer states create a **bias–variance / information–sparsity trade-off**.

In [ ]:
humidity_cut = train_raw["humidity"].median()

def add_composite_state(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["temp_state"] = assign_temperature_state(out["meantemp"]).astype(str)

    out["humidity_state"] = np.where(
        out["humidity"] <= humidity_cut,
        "LowerHumidity",
        "HigherHumidity",
    )

    out["composite_state"] = (
        out["temp_state"] + " | " + out["humidity_state"]
    )

    return out

train_composite = add_composite_state(train_raw)
composite_states = sorted(train_composite["composite_state"].unique())

display(
    train_composite[
        ["date", "meantemp", "humidity", "composite_state"]
    ].head()
)

print("Composite states:", len(composite_states))
print(*composite_states, sep="\n")

### Why create a richer six-state representation?

A temperature-only state may omit relevant information.

The composite state adds humidity:

\[
Z_t=(\text{temperature regime},\text{humidity regime}).
\]

### Inference

A richer state can make the first-order Markov assumption more plausible because the state contains more information.

But there is a cost: more states mean more transition parameters and fewer observations per transition. This is a classic **model-complexity versus data-sparsity trade-off**.

# 22. Exercises

## Exercise 1 — six-state chain

Using `train_composite`:

1. estimate the six-state transition matrix,
2. visualise it,
3. identify communicating classes,
4. determine irreducibility,
5. determine periods,
6. calculate the stationary distribution,
7. compare held-out NLL against the three-state model.

Ask whether improved state information justifies the larger parameter space.

---

## Exercise 2 — reverse hitting problem

Calculate:

\[
E_{\text{Hot}}[T_{\text{Cold}}],
\qquad
E_{\text{Mild}}[T_{\text{Cold}}].
\]

Compare these with the hitting times to `Hot`.

Explain the asymmetry using transition probabilities.

In [ ]:
# Exercise 2 starter:
#
# cold_hitting = mc.expected_hitting_times("Cold")
# display(cold_hitting)

### Exercise purpose

This exercise reverses the target from `Hot` to `Cold`.

The goal is to discover that hitting times need not be symmetric:

\[
E_{\text{Cold}}[T_{\text{Hot}}]
\neq
E_{\text{Hot}}[T_{\text{Cold}}].
\]

Why? Because the transition matrix is generally not symmetric.

## Exercise 3 — verify Chapman–Kolmogorov

Take \(m=3\) and \(n=5\).

Numerically verify

\[
P^{m+n}=P^mP^n.
\]

In [ ]:
m, n = 3, 5

lhs = mc.n_step(m + n)
rhs = mc.n_step(m) @ mc.n_step(n)

print("max |P^(m+n) - P^m P^n| =", np.max(np.abs(lhs - rhs)))

### Interpreting the Chapman–Kolmogorov check

The maximum absolute error should be zero or extremely close to machine precision.

That numerically verifies

\[
P^{m+n}=P^mP^n.
\]

The tiny discrepancy, if any, is floating-point arithmetic—not a failure of the theorem.

## Exercise 4 — stationary does not imply convergence

Consider

\[
P=
\begin{bmatrix}
0&1\\
1&0
\end{bmatrix}.
\]

1. calculate its stationary distribution,
2. calculate its period,
3. inspect \(P^n\) for \(n=1,\ldots,6\),
4. explain why \(P^n\) does not converge.

In [ ]:
periodic_mc = DiscreteMarkovChain(
    states=("A", "B"),
    P=np.array([
        [0.0, 1.0],
        [1.0, 0.0],
    ]),
)

print("Stationary distribution:", periodic_mc.stationary_distribution())
print("Periods:", periodic_mc.periods(max_n=20))

for n in range(1, 7):
    print(f"\nP^{n}")
    print(periodic_mc.n_step(n))

### What this counterexample teaches

The two-state chain alternates deterministically:

\[
A\to B\to A\to B\to\cdots
\]

It has stationary distribution

\[
\pi=(1/2,1/2),
\]

but \(P^n\) oscillates forever.

### Inference

**Existence of a stationary distribution does not by itself imply convergence of \(P^n\).**

Aperiodicity is the missing condition.

## Exercise 5 — Monte Carlo hitting times

For `Hot` as the target:

1. simulate thousands of trajectories from `Cold`,
2. record first hitting times,
3. estimate the mean,
4. compare with the analytical solution from \((I-Q)h=\mathbf1\),
5. plot the empirical hitting-time distribution in Bokeh.

This is a good way to connect theoretical expectations with Monte Carlo estimation.

# 23. Final concept map

The real-data workflow was:

\[
\boxed{\text{chronological observations}}
\]

\[
\downarrow
\]

\[
\boxed{\text{define }X_t\text{ and }S}
\]

\[
\downarrow
\]

\[
\boxed{
\hat p_{ij}
=
\frac{N_{ij}}{\sum_kN_{ik}}
}
\]

\[
\downarrow
\]

\[
\boxed{P,\;P^2,\;P^n}
\]

\[
\downarrow
\]

\[
\boxed{
\text{communication}
+
\text{irreducibility}
+
\text{periodicity}
}
\]

\[
\downarrow
\]

\[
\boxed{\text{recurrence / transience}}
\]

\[
\downarrow
\]

\[
\boxed{\pi P=\pi}
\]

\[
\downarrow
\]

\[
\boxed{P^n\to\mathbf1\pi}
\]

while first-step analysis gives

\[
\boxed{
\text{hitting probabilities / times}
\rightarrow
\text{absorbing chains}
\rightarrow
N=(I-Q)^{-1}
}.
\]

# 24. Main lessons

### Mathematical
- A transition matrix is a collection of conditional probability distributions.
- Multi-step behaviour comes from powers of \(P\).
- Communicating classes connect Markov chains with graph theory.
- Finite irreducible chains are positive recurrent.
- Aperiodicity is essential for ordinary limiting convergence.
- Stationary distributions solve an eigenvector problem.
- First-passage times solve linear systems.
- Absorbing chains introduce the fundamental matrix.

### Modelling
- State design determines whether the Markov approximation is sensible.
- Time homogeneity should be checked, not assumed.
- Real data can violate textbook assumptions.
- Higher-order dependence can reveal inadequate state representation.
- Stochastic models should be evaluated on chronological held-out data.

### Data-science connection
These ideas lead directly toward:
- PageRank,
- hidden Markov models,
- MCMC,
- queueing models,
- reliability models,
- random walks on graphs,
- Markov decision processes and reinforcement learning.

# Beginner revision checklist — can you explain these without code?

Before considering the notebook complete, make sure you can answer each question in words.

### State construction
1. What exactly is \(X_t\)?
2. Why did we discretise temperature?
3. Why were thresholds learned only from training data?

### Transition matrix
4. What does one row of \(P\) represent?
5. Why must every row sum to 1?
6. What does a large diagonal probability mean?

### Multi-step transitions
7. What does \((P^7)_{ij}\) mean?
8. Why is \(P^{m+n}=P^mP^n\)?

### Graph theory / NetworkX
9. Why is the graph directed?
10. What is an edge weight?
11. What is accessibility?
12. What is a communicating class?
13. Why are SCCs the graph equivalent of communicating classes?
14. What does `nx.is_strongly_connected(G)` tell us?
15. Why is weak connectivity insufficient?
16. How do cycles relate to periodicity?

### Long-run behaviour
17. What is recurrence?
18. What does irreducibility imply for a finite chain?
19. What does period 1 mean?
20. What does \(\pi P=\pi\) mean?
21. Why does stationarity not automatically imply convergence?

### First passage
22. What is \(T_j\)?
23. Why does first-step analysis produce linear equations?
24. What does the absorbing fundamental matrix \(N=(I-Q)^{-1}\) mean?

### Model adequacy
25. How did we check whether one lag is enough?
26. How did we investigate time homogeneity?
27. Why can real weather violate the fitted Markov assumptions?

If you can explain these clearly, you understand the conceptual core of the notebook rather than merely being able to run it.

# Final perspective

A stochastic-process analysis is not:

> "I calculated a transition matrix, therefore the system is Markov."

The correct reasoning is:

\[
\text{define state}
\rightarrow
\text{estimate transitions}
\rightarrow
\text{analyse mathematical consequences}
\rightarrow
\text{check assumptions against data}
\rightarrow
\text{state conclusions with appropriate scope}.
\]

That final step — distinguishing what the **model implies** from what the **data prove** — is one of the most important habits to develop when studying stochastic processes.